In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf
import baostock as bs

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,peTTM,pbMRQ,psTTM,pcfNcfTTM,isST
date,,,,,,,,,,,,,,,,,
2010-01-04,sh.600000,64.829140,64.947929,62.839423,62.928515,64.413379,66191338,1.419984e+09,1,0.835129,1,-2.3052,14.435065,2.877081,5.316299,7.091240,0
2010-01-05,sh.600000,63.581855,64.086709,61.740624,63.403671,62.928515,115147943,2.436891e+09,1,1.452808,1,0.7551,14.287342,2.898805,5.178308,7.144784,0
2010-01-06,sh.600000,63.225488,63.255185,62.007900,62.156386,63.403671,96782575,2.034174e+09,1,1.221095,1,-1.9672,14.006279,2.841779,5.076439,7.004231,0
2010-01-07,sh.600000,62.007900,62.483056,60.285458,60.760614,62.156386,85236072,1.761801e+09,1,1.075414,1,-2.2456,13.691757,2.777965,4.962444,6.846945,0
2010-01-08,sh.600000,60.404247,61.770322,60.285458,61.443652,60.760614,65707646,1.349532e+09,1,0.829026,1,1.1241,13.845672,2.809193,5.018229,6.923915,0


In [2]:
dt.dtypes

code            object
open           float64
high           float64
low            float64
close          float64
preclose       float64
volume           int64
amount         float64
adjustflag       int64
turn           float64
tradestatus      int64
pctChg         float64
peTTM          float64
pbMRQ          float64
psTTM          float64
pcfNcfTTM      float64
isST             int64
dtype: object

In [3]:
# 登陆系统
lg = bs.login()
rs = bs.query_stock_industry()
# rs = bs.query_stock_basic(code_name="浦发银行")
print('query_stock_industry error_code:'+rs.error_code)
print('query_stock_industry respond  error_msg:'+rs.error_msg)

# 打印结果集
industry_list = []
while (rs.error_code == '0') & rs.next():
    # 获取一条记录，将记录合并在一起
    industry_list.append(rs.get_row_data())
result = pd.DataFrame(industry_list, columns=rs.fields)

login success!
query_stock_industry error_code:0
query_stock_industry respond  error_msg:success


In [4]:
result.head()

,updateDate,code,code_name,industry,industryClassification
0,2026-02-02,sh.600000,浦发银行,J66货币金融服务,证监会行业分类
1,2026-02-02,sh.600001,邯郸钢铁,,证监会行业分类
2,2026-02-02,sh.600002,齐鲁石化,,证监会行业分类
3,2026-02-02,sh.600003,ST东北高,,证监会行业分类
4,2026-02-02,sh.600004,白云机场,G56航空运输业,证监会行业分类


In [16]:
# 滚动市盈率，这里用最后一日的数据
peTTMs = {} # 这个是一个字典
with tqdm(total=len(result)) as pbar:
    for index, row in result.iterrows():
        code = row['code']
        industry = row['industry']
        # 然后取得这个股票的最后数据
        dt = datasource.get_data(code)
        pbar.update(1)
        if len(dt) == 0:
            continue
        # 我需要取得最后的动态市盈率，然后我要确认这个是还在正常交易的
        _tradestatu_index = list(dt.columns).index('tradestatus')
        _tradestatus = dt.iat[len(dt)-1, _tradestatu_index]
        _peTTM = dt.iloc[-1,:]['peTTM']
        if _tradestatus == 1:
            if industry not  in peTTMs:
                peTTMs[industry] = []
            peTTMs[industry].append(_peTTM.item())
            

100%|██████████████████████████████████████████████████████████████████████████████| 5501/5501 [01:32<00:00, 59.38it/s]


In [17]:
for i in peTTMs:
    # 这里将所有的计算
    lst = peTTMs[i]
    _mean = sum(lst)/len(lst)
    print(f'{i}平均市盈率:{_mean}')

J66货币金融服务平均市盈率:9.020923733333333
G56航空运输业平均市盈率:24.54613735714286
C36汽车制造业平均市盈率:80.10540096410256
K70房地产业平均市盈率:0.10726848863636372
D46水的生产和供应业平均市盈率:51.14941123809523
C31黑色金属冶炼和压延加工业平均市盈率:7.54626964516129
D44电力、热力生产和供应业平均市盈率:25.92563877173913
G54道路运输业平均市盈率:385.6465184193549
G55水上运输业平均市盈率:-124.72564517142858
B07石油和天然气开采业平均市盈率:-13.145321000000001
J67资本市场服务平均市盈率:40.74053325
C35专用设备制造业平均市盈率:-210.01640652676053
I63电信、广播电视和卫星传输服务平均市盈率:24.109409428571432
C37铁路、船舶、航空航天和其他运输设备制造业平均市盈率:66.73320214634145
E48土木工程建筑业平均市盈率:12.175781426470587
F51批发业平均市盈率:-604.284677204301
R87广播、电视、电影和录音制作业平均市盈率:-193.39790276190476
N78公共设施管理业平均市盈率:28.127650291666665
L72商务服务业平均市盈率:7.028002972222222
C15酒、饮料和精制茶制造业平均市盈率:17.247026
C39计算机、通信和其他电子设备制造业平均市盈率:29.63857418981482
C27医药制造业平均市盈率:5.92152410714286
C26化学原料和化学制品制造业平均市盈率:21.088301365714283
C38电气机械和器材制造业平均市盈率:20.361707908536584
C40仪器仪表制造业平均市盈率:35.88275461728395
C13农副食品加工业平均市盈率:73.00628304761905
C20木材加工和木、竹、藤、棕、草制品业平均市盈率:-19.687707538461538
A04渔业平均市盈率:-18.94918783333333
C2